In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.13 Semiconductors: Fermi–Dirac in a Gap

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.13",
    title="Semiconductors: Fermi–Dirac in a Gap",
    blurb="Nothing new happens in this notebook — that is its point. The Fermi function of "
    "§7.7, the integrals of §7.3, and the thermal wavelength of §7.8 are aimed at the "
    "gapped density of states built in §7.12, and an entire technology falls out: a "
    "law of mass action that doping cannot touch, ten billion carriers per cubic "
    "centimetre computed from Planck's constant and one energy gap, a temperature "
    "plateau on which every chip you own depends, and — in the heavily doped limit — "
    "a return, on schedule, to the metal where the movement began.",
    difficulty="advanced",
    estimate="195–235 min",
)

## Notebook overview

[§7.12](bloch-theorem-band-structure.ipynb) ended with a promise: set the chemical potential loose in the gap and let Fermi–Dirac do
the rest. This notebook keeps that promise with conspicuous economy, because every tool in it
is second-hand. The distribution is the Fermi function of [§7.7](bose-einstein-fermi-dirac.ipynb), unchanged; the carrier integrals are
the $F_{1/2}$ machinery of [§7.3](statistical-toolkit.ipynb) pointed at a new density of states; the criterion that decides when
they collapse to Boltzmann form is verbatim the $n\lambda_T^3 \ll 2$ of [§7.8](classical-limit-thermal-wavelength.ipynb); and the one genuinely
new object, a density of states with a gap in it, was built, gated, and handed over by [§7.12](bloch-theorem-band-structure.ipynb).
The statistics resumes here, finishes Movement III, and an entire technology falls out of the
resumption.

The model is two parabolic bands: conduction states above $E_c$ with density-of-states mass
$m_c$, valence states below $E_v$ with $m_v$, separated by the gap $E_g$ (silicon: 1.12 eV;
the masses are the band curvatures of [§7.12](bloch-theorem-band-structure.ipynb), taken here as stated inputs the way a DFT calculation
would supply them). Electrons in the conduction band are counted by $n_F$; **holes**, the negative-mass band-top carriers of [§7.12](bloch-theorem-band-structure.ipynb) now made statistical, are counted by $1 - n_F$ in the
valence band: two dilute gases, one Fermi function. In the Boltzmann limit the engineer's
workhorse constant appears, $N_c = 2(m_c k_BT/2\pi\hbar^2)^{3/2}$, and turns out to be
*literally* $2/\lambda_T^3$ — the thermal wavelength of [§7.8](classical-limit-thermal-wavelength.ipynb) wearing overalls — with the
nondegeneracy license quantified against the exact integrals (ratio 1.000 at
$E_c - \mu = 200$ meV, 1.05 at 50 meV, 2.5 inside the band). Multiplying the two populations
cancels $\mu$ and yields the **law of mass action** $np = n_i^2$: however doping tilts the
populations, their product is fixed by gap and temperature alone. The data jewel follows —
$n_i(\text{Si}, 300\,\text{K}) = 8.9\times10^9\ \text{cm}^{-3}$ from $\hbar$, one gap, and two
masses, within 10% of the accepted value; a Ge/GaAs table spanning seven decades from gaps
that differ by a factor of two; one carrier pair per $5.6\times10^{12}$ silicon atoms.

The centerpiece is **doping**. Donor statistics and exact charge neutrality (solved by
`scipy.optimize.brentq` at every temperature) produce the classic Arrhenius portrait with its
three regimes: freeze-out, the **saturation plateau** ($n \approx N_D$ across 100–450 K, the reason electronics works at all: carrier density set by chemistry rather than temperature),
and the intrinsic takeover, with device death computed at 719 K for $N_D = 10^{16}$. The
mass-action seesaw meanwhile crushes the minority holes to $7.9\times10^3\ \text{cm}^{-3}$,
twelve decades below the majority. Then the license is revoked on schedule: pushed toward
$N_c = 2.8\times10^{19}\ \text{cm}^{-3}$ the carriers go degenerate, $\mu$ enters the band,
and the heavily doped semiconductor *is* the metal of [§7.9](fermi-gas-zero-temperature.ipynb), and Movement III closes its loop on the
same dimensionless number that opened it. A stretch computes the most famous number in hobby
electronics: $V_{bi} = (k_BT/q)\ln(N_AN_D/n_i^2) = 0.720$ V, the silicon diode drop, from one
logarithm of mass action, with the junction's mechanics deferred outward, per the volume's
contract.

> **Conventions (this notebook).** Energies in eV, densities in cm⁻³, $E_c = 0$ as the energy
> reference (so $E_v = -E_g$ and everything in the gap is negative); $k_BT = 0.02585$ eV at
> 300 K. Unit discipline is *the* numerical trap here (a silently mixed eV/J or cm⁻³/m⁻³ shifts $n_i$ by ten orders of magnitude), so every Joule↔eV and m⁻³↔cm⁻³ conversion happens
> inside exactly two Setup helpers (`N_eff`, `lambda_T`) and nowhere else. The exact carrier
> integrals use `scipy.integrate.quad` on the $F_{1/2}$ form with the substitution
> $x = (\varepsilon - E_c)/k_BT$ and a stated cutoff (insensitivity shown once); every
> neutrality condition (intrinsic, doped, and the takeover temperature) is solved by
> `scipy.optimize.brentq` with its bracket stated. Band parameters carry their sources.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the hole integral against a brute-force second route; the $N_c = 2/
> \lambda_T^3$ identity at $10^{-10}$; the silicon number against the accepted range (cited,
> not tuned); the intrinsic $\mu$ formula against the exact solve to all displayed digits;
> the three-regime portrait against its landmark values; the degenerate ladder against the
> criterion of [§7.8](classical-limit-thermal-wavelength.ipynb); the diode drop against the number every hobbyist knows. A ✓ is strong evidence;
> a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope (the framing constraint, binding).** Nothing new happens here, and the visible prose
> says so: this is quantum statistical mechanics consuming the object of [§7.12](bloch-theorem-band-structure.ipynb), not device physics.
> The p–n junction's mechanics (depletion, rectification, the transistor) are *named and
> deferred* (Sze & Ng, *Physics of Semiconductor Devices*); real gaps and masses from
> electronic structure belong to DFT and the MMM course; the Saha equation is named as this
> notebook's mathematics in a stellar atmosphere. See Ashcroft & Mermin (Ch. 28); Kittel
> (Ch. 8). Cross-reference [§7.7](bose-einstein-fermi-dirac.ipynb) (the function, unchanged), [§7.3](statistical-toolkit.ipynb) (the integrals), [§7.8](classical-limit-thermal-wavelength.ipynb) (the
> license, twice decisive), [§7.9](fermi-gas-zero-temperature.ipynb) (the degenerate return), [§7.12](bloch-theorem-band-structure.ipynb) (the object consumed), [§6.20](../06-quantum-mechanics/identical-particles.ipynb)
> (Pauli underneath it all), and forward to [§7.14](photon-gas-planck.ipynb) (the photon gas opens Movement IV).

## Theory in brief

### Two bands, two species

Near its extrema, the band structure of [§7.12](bloch-theorem-band-structure.ipynb) is parabolic: conduction states with curvature mass
$m_c$ above $E_c$, valence states with $m_v$ below $E_v$, and *no states at all* in the gap
$E_g = E_c - E_v$. Each band therefore carries a √-edge density of states, and the Fermi
function of [§7.7](bose-einstein-fermi-dirac.ipynb) counts both species at once:

```{math}
:label: eq-two-bands
n = \int_{E_c}^{\infty} g_c(\varepsilon)\,n_F(\varepsilon)\,d\varepsilon,
\qquad
p = \int_{-\infty}^{E_v} g_v(\varepsilon)\,\bigl[1 - n_F(\varepsilon)\bigr]\,d\varepsilon .
```

The hole bookkeeping is the filled-band argument of [§7.12](bloch-theorem-band-structure.ipynb) made statistical: a full valence band is
inert, so only its *empty* states, with weight $1 - n_F$, respond, and each responds as a positive
carrier of mass $|m_v|$. Exact evaluation goes through the substitution of [§7.3](statistical-toolkit.ipynb)
$x = (\varepsilon - E_c)/k_BT$, which turns each integral into the Fermi–Dirac form
$F_{1/2}(\eta) = \int_0^\infty \sqrt{x}\,dx/(1 + e^{x-\eta})$, evaluated by
`scipy.integrate.quad` with a stated cutoff.

### The Boltzmann limit, and an old friend in overalls

When $E_c - \mu \gg k_BT$ the Fermi tail is exponential and the integral collapses:

```{math}
:label: eq-boltzmann-carriers
n \;\longrightarrow\; N_c\,e^{-(E_c-\mu)/k_BT},
\qquad
N_c = 2\left(\frac{m_c k_BT}{2\pi\hbar^2}\right)^{3/2} \equiv \frac{2}{\lambda_T^3},
```

and likewise $p \to N_v e^{-(\mu - E_v)/k_BT}$. The identity on the right deserves a pause:
the "effective density of states," the workhorse constant of every device textbook, is
*identically* $2/\lambda_T^3$ — the thermal wavelength of [§7.8](classical-limit-thermal-wavelength.ipynb) for the carrier mass, spin included
(verified below: $2.816\times10^{19}\ \text{cm}^{-3}$ by both routes). Nondegeneracy for
carriers is therefore verbatim the criterion of [§7.8](classical-limit-thermal-wavelength.ipynb) $n\lambda_T^3 \ll 2$, i.e. $n \ll N_c$ — the
movement's opening map ruling its closing notebook. The license is precise and its revocation
is scheduled: quantified against the exact integrals, Boltzmann/exact = 1.000 at
$E_c - \mu = 200$ meV, 1.050 at 50 meV, and 2.5 once $\mu$ sits 50 meV *inside* the band.

### The law of mass action

Multiply the two Boltzmann populations and watch $\mu$ cancel:

```{math}
:label: eq-mass-action
np = N_cN_v\,e^{-E_g/k_BT} \equiv n_i^2(T) .
```

The product is independent of $\mu$, and therefore of *doping*: however the populations are
tilted, their product is fixed by the gap and the temperature alone. The structure is a
chemical equilibrium constant for the reaction $\varnothing \rightleftharpoons e^- + h^+$,
and the same mathematics, applied to $\text{H} \rightleftharpoons p + e^-$ in a stellar
atmosphere, is the Saha ionization equation: one breath, outward.

### The silicon number, and seven decades

Setting $n = p$ turns mass action into the intrinsic density in one square root, and
differentiating its logarithm with respect to the gap measures the exponential's leverage:

```{math}
:label: eq-intrinsic-density
n_i = \sqrt{N_cN_v}\;e^{-E_g/2k_BT},
\qquad
\frac{d\ln n_i}{dE_g} = -\frac{1}{2k_BT}
\;\;(\text{a factor of 10 per 119 meV at 300 K}).
```

For silicon ($E_g = 1.12$ eV, $m_c = 1.08$, $m_v = 0.81$) this gives
$8.9\times10^9\ \text{cm}^{-3}$ at 300 K against the accepted $9.65\times10^9$ (modern) to
$1.45\times10^{10}$ (older texts): within 10% from $\hbar$, one gap, and two masses, the
residual honestly assigned to the effective-mass inputs rather than tuned away. The same three
constants give germanium ($0.67$ eV) $1.5\times10^{13}$ and GaAs ($1.42$ eV) $2.2\times10^6$:
**seven decades** of carrier density from gaps within a factor of two, because the
sensitivity is exponential, the field's defining fact. And the scale: one carrier pair per
$5.6\times10^{12}$ silicon atoms, a part-per-trillion impurity of pure thermodynamics.

### The chemical potential of an insulator

Intrinsic neutrality $n = p$, solved exactly by `brentq` on the Fermi–Dirac integrals, sits at

```{math}
:label: eq-intrinsic-mu
\mu_i = \frac{E_c + E_v}{2} + \frac{3}{4}k_BT\ln\frac{m_v}{m_c},
```

mid-gap plus a mass drift (the heavier band offers more states; $\mu$ leans away from it) —
and the Boltzmann-level formula matches the exact solve to all displayed digits
($-0.56558$ eV at 300 K). Pause on the scandal [§7.7](bose-einstein-fermi-dirac.ipynb) normalized: the chemical potential of an
insulator lives *where there are no states at all*. That is routine for the formalism ($\mu$ is the bookkeeping price of one more particle, the lineage of [§7.4](thermal-density-matrix.ipynb), not the address of any occupied level) and offensive only to the intuition that expects $\mu$ to live on a state.

### Doping: the three regimes

Donors (phosphorus in silicon) park an electron $\Delta E_D = 45$ meV below $E_c$; the level
ionizes per the degeneracy-2 donor statistics, and neutrality picks $\mu$:

```{math}
:label: eq-doping-regimes
N_D^+ = \frac{N_D}{1 + 2\,e^{(\mu - E_D)/k_BT}},
\qquad
n = p + N_D^+ \quad(\text{solved by brentq at each } T).
```

The classic Arrhenius portrait has three acts ($N_D = 10^{16}$): **freeze-out** below
$\sim$80 K (carriers recaptured by their donors, $n$ collapsing with slope $\Delta E_D/2$);
the **saturation plateau** from $\sim$100–450 K, where $n \approx N_D$ — read it slowly: for
three hundred kelvin the carrier density is set by *chemistry*, not temperature, and this
plateau is the operating premise of all electronics; and the **intrinsic takeover** as
$n_i(T)$ storms past $N_D$: device death at 719 K, and the reason power electronics reaches
for wide gaps (SiC, GaN: one breath, outward). Meanwhile the mass-action seesaw does its
quiet work: doping to $10^{16}$ crushes the minority holes to $n_i^2/N_D = 7.9\times10^3\
\text{cm}^{-3}$, twelve decades down, the leverage on which bipolar devices run.

### The license revoked: the degenerate return

The Boltzmann license was granted by the criterion of [§7.8](classical-limit-thermal-wavelength.ipynb), so it must be audited by the same
number; in this notebook's variables the degeneracy parameter is simply $n/N_c$, marched
here up the doping ladder:

```{math}
:label: eq-degenerate-return
\frac{n\lambda_T^3}{2} = \frac{n}{N_c}:
\qquad
3.6\times10^{-10} \;(\text{intrinsic})
\;\to\; 3.6\times10^{-4} \;(10^{16})
\;\to\; 1 \;(N_c = 2.8\times10^{19})
\;\to\; 3.6 \;(10^{20}).
```

At $n \sim N_c$ Boltzmann fails (the ratios of {eq}`eq-boltzmann-carriers` now bite), $\mu$
crosses *into* the conduction band, and the heavily doped semiconductor is a metal in every
respect the movement owns: a Fermi level in a band, a degenerate gas, the formulas of [§7.9](fermi-gas-zero-temperature.ipynb).
Industry uses exactly this limit daily (degenerate emitters, ohmic contacts). Movement III
ends where it began: the dimensionless number of the map of [§7.8](classical-limit-thermal-wavelength.ipynb) adjudicates its last notebook.

### The 0.7 volts (stretch)

Join an n-type and a p-type region in equilibrium and *one* chemical potential must span both
(the equilibrium condition of [§7.4](thermal-density-matrix.ipynb) and [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb)), so the bands bend by the difference of the bulk
chemical potentials:

```{math}
:label: eq-builtin-potential
qV_{bi} = k_BT\,\ln\frac{N_AN_D}{n_i^2} = 0.720\ \text{eV}
\quad(N_A = N_D = 10^{16}\ \text{cm}^{-3},\ 300\ \text{K}),
```

the ~0.7 volts every silicon diode drops, computed from one logarithm of mass action. The
junction's mechanics, depletion widths and rectification and the transistor, belong to device
physics (Sze & Ng) and are deferred outward, per the framing constraint: this course computes
the number and stops.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: energies in eV, densities in cm^-3, E_c = 0 (so
# E_v = -E_g). ALL Joule↔eV and m^-3↔cm^-3 conversion happens inside N_eff and
# lambda_T below and nowhere else — unit discipline is this notebook's stated trap
# (a silently mixed unit shifts n_i by ten orders of magnitude).
KB = 8.617333262e-5  # Boltzmann constant, eV/K (CODATA 2018)
from scipy.constants import m_e as ME  # electron mass, kg (CODATA 2018)
from scipy.constants import h as H_PLANCK  # Planck constant, J·s (exact, SI 2019)
from scipy.constants import eV as EV  # eV in J (exact, SI 2019)

# ħ DERIVED from the exact h rather than typed as its own rounded literal: a
# 10-digit ħ alongside the exact h breaks h = 2πħ at the 6e-10 level, which is
# enough to fail Exercise 2's 1e-10 identity gate between the two routes.
HBAR = H_PLANCK / (2.0 * np.pi)  # reduced Planck constant, J·s

# Band parameters (density-of-states masses in units of m_e; 300 K gaps).
# Sources: Sze & Ng, Physics of Semiconductor Devices, Ch. 1 (Si, Ge, GaAs);
# the masses are the band-curvature inputs whose origin is §7.12 (and, for real
# materials, a DFT band structure — the outward provenance).
SI = {"Eg": 1.12, "mc": 1.08, "mv": 0.81, "name": "Si"}
GE = {"Eg": 0.67, "mc": 0.56, "mv": 0.29, "name": "Ge"}
GAAS = {"Eg": 1.42, "mc": 0.067, "mv": 0.47, "name": "GaAs"}
E_C = 0.0  # conduction band edge: the energy reference
DE_D = 0.045  # donor depth ΔE_D below E_c (phosphorus in silicon), eV
G_D = 2  # donor spin degeneracy (one electron, two spins — the honest factor)
N_ATOMS_SI = 5.0e22  # silicon atom density, cm^-3
X_CUT = 60.0  # baseline cutoff for the F_{1/2} integrals (insensitivity shown in Ex 1)


def fermi_half(eta, x_cut=X_CUT):
    """The Fermi–Dirac integral F_{1/2}(η) = ∫_0^∞ √x dx / (1 + e^{x−η}).

    Evaluated by scipy.integrate.quad on [0, cutoff] with cutoff = max(x_cut,
    η + x_cut): for nondegenerate η the tail beyond x = 60 is e^{−60} of the
    integrand (insensitivity gated in Exercise 1), and for degenerate η > 0 the
    window must extend past the Fermi step, hence the η + x_cut guard. The
    tolerance is PURELY RELATIVE (epsabs = 0): deep in the gap F_{1/2} ~ e^η can
    sit far below quad's default absolute floor of 1.5e-8, and leaving that
    default in place silently degrades the integral to ~1e-3 relative accuracy —
    the unit trap's quieter sibling, caught by Exercise 1's cutoff gate. This is
    the integral of §7.3, unchanged — the notebook's contract is that nothing here is new.

    Parameters
    ----------
    eta : float
        Reduced chemical potential (μ − E_edge)/k_BT measured into the band.
    x_cut : float, optional
        Baseline integration cutoff (default 60).

    Returns
    -------
    float
        F_{1/2}(η).
    """
    upper = max(x_cut, eta + x_cut)
    val, _ = quad(
        lambda x: np.sqrt(x) / (1.0 + np.exp(x - eta)),
        0.0,
        upper,
        limit=200,
        epsabs=0.0,
        epsrel=1e-11,
    )
    return val


def N_eff(m_rel, T):
    """The effective density of states N = 2(m k_BT / 2πħ^2)^{3/2}, in cm^-3.

    The engineer's workhorse constant — and, per Exercise 2's identity, literally
    2/λ_T^3: the thermal wavelength of §7.8 for the carrier mass, spin included. This
    helper is one of the notebook's two unit-conversion points: k_BT enters in
    Joules (KB·T·EV) and the m^-3 result is converted to cm^-3 once, here.

    Parameters
    ----------
    m_rel : float
        Density-of-states mass in units of the electron mass.
    T : float
        Temperature in K.

    Returns
    -------
    float
        Effective density of states, cm^-3.
    """
    kT_J = KB * T * EV  # the eV→J conversion point
    n_per_m3 = 2.0 * (m_rel * ME * kT_J / (2.0 * np.pi * HBAR**2)) ** 1.5
    return n_per_m3 * 1e-6  # the m^-3 → cm^-3 conversion point


def lambda_T(m_rel, T):
    """The thermal de Broglie wavelength λ_T = h/√(2π m k_BT), in cm.

    Verbatim the definition of §7.8, evaluated for a carrier of mass m_rel·m_e — the
    second of the two unit-conversion points (J and m inside, cm out), so that
    2/λ_T^3 lands directly in cm^-3 for Exercise 2's identity gate.

    Parameters
    ----------
    m_rel : float
        Carrier mass in units of the electron mass.
    T : float
        Temperature in K.

    Returns
    -------
    float
        Thermal wavelength, cm.
    """
    kT_J = KB * T * EV
    lam_m = H_PLANCK / np.sqrt(2.0 * np.pi * m_rel * ME * kT_J)
    return lam_m * 1e2  # m → cm


def n_electrons(mu, T, mat=SI):
    """Exact conduction-band electron density n(μ, T) by the Fermi–Dirac integral.

    n = N_c(T) · (2/√π) · F_{1/2}(η) with η = (μ − E_c)/k_BT — eq-two-bands after
    the substitution x = (ε − E_c)/k_BT, evaluated by scipy.integrate.quad via
    fermi_half. Exact at every degeneracy (the Boltzmann form is Exercise 2's
    limit, not an input).

    Parameters
    ----------
    mu : float
        Chemical potential, eV (E_c = 0 reference).
    T : float
        Temperature in K.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Electron density, cm^-3.
    """
    kT = KB * T
    eta = (mu - E_C) / kT
    return N_eff(mat["mc"], T) * (2.0 / np.sqrt(np.pi)) * fermi_half(eta)


def p_holes(mu, T, mat=SI):
    """Exact valence-band hole density p(μ, T) by the mirrored Fermi–Dirac integral.

    p = N_v(T) · (2/√π) · F_{1/2}(η_p) with η_p = (E_v − μ)/k_BT: the weight
    1 − n_F(ε) of an empty valence state equals n_F mirrored through the edge
    (1 − 1/(1+e^a) = 1/(1+e^{−a})), so holes are counted by the same integral as
    electrons with energies measured downward from E_v. Exercise 1 gates this
    algebra against a brute-force second route with no substitution.

    Parameters
    ----------
    mu : float
        Chemical potential, eV (E_c = 0 reference).
    T : float
        Temperature in K.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Hole density, cm^-3.
    """
    kT = KB * T
    E_v = E_C - mat["Eg"]
    eta_p = (E_v - mu) / kT
    return N_eff(mat["mv"], T) * (2.0 / np.sqrt(np.pi)) * fermi_half(eta_p)


def n_boltzmann(mu, T, mat=SI):
    """Boltzmann-limit electron density n = N_c e^{−(E_c−μ)/k_BT} (eq-boltzmann-carriers).

    The nondegenerate approximation whose license — n ≪ N_c, verbatim the
    nλ^3 ≪ 2 of §7.8 — Exercise 2 quantifies and Exercise 6 revokes. The exponent is
    negative throughout this notebook's use (μ below E_c), so no overflow guard is
    needed on this branch; the low-T guard lives in the neutrality solver instead.

    Parameters
    ----------
    mu : float
        Chemical potential, eV.
    T : float
        Temperature in K.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Boltzmann electron density, cm^-3.
    """
    return N_eff(mat["mc"], T) * np.exp(-(E_C - mu) / (KB * T))


def p_boltzmann(mu, T, mat=SI):
    """Boltzmann-limit hole density p = N_v e^{−(μ−E_v)/k_BT} (eq-boltzmann-carriers).

    Parameters
    ----------
    mu : float
        Chemical potential, eV.
    T : float
        Temperature in K.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Boltzmann hole density, cm^-3.
    """
    E_v = E_C - mat["Eg"]
    return N_eff(mat["mv"], T) * np.exp(-(mu - E_v) / (KB * T))


def mu_intrinsic(T, mat=SI):
    """The intrinsic chemical potential: n(μ) = p(μ) solved by scipy.optimize.brentq.

    The bracket is the gap itself, [E_v + δ, E_c − δ]: at the valence edge holes
    dominate (n − p < 0), at the conduction edge electrons do (n − p > 0), and
    n − p is monotone in μ, so the root is unique. Exercise 4 gates this exact
    solve against the Boltzmann-level formula mid-gap + (3/4)k_BT ln(m_v/m_c).

    Parameters
    ----------
    T : float
        Temperature in K.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Intrinsic chemical potential μ_i, eV.
    """
    E_v = E_C - mat["Eg"]
    return brentq(
        lambda mu: n_electrons(mu, T, mat) - p_holes(mu, T, mat),
        E_v + 1e-9,
        E_C - 1e-9,
        xtol=1e-12,
    )


def n_intrinsic(T, mat=SI):
    """The intrinsic carrier density n_i(T): the electron density at μ = μ_i(T).

    Computed from the exact machinery (brentq neutrality, then the Fermi–Dirac
    integral) rather than the closed form √(N_cN_v)e^{−E_g/2k_BT}, so that
    Exercise 3's comparison of the two is a genuine check and Exercise 5's takeover
    solve inherits no Boltzmann assumption.

    Parameters
    ----------
    T : float
        Temperature in K.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Intrinsic carrier density, cm^-3.
    """
    return n_electrons(mu_intrinsic(T, mat), T, mat)


def solve_neutrality(T, N_D, dE_D=DE_D, mat=SI):
    """Doped charge neutrality n = p + N_D^+ solved for μ by scipy.optimize.brentq.

    N_D^+ = N_D/(1 + 2 e^{(μ−E_D)/k_BT}) is the degeneracy-2 donor occupation
    (eq-doping-regimes; omitting the g = 2 shifts freeze-out visibly — Exercise 5
    comments on it). Bracket [E_v + δ, E_c + 0.3]: at the valence edge the hole
    term makes n − p − N_D^+ hugely negative; 0.3 eV above E_c the electron
    integral dominates everything at any T in the sweep — a bracket that survives
    both the 30 K freeze-out and the near-degenerate top of the sweep (the stated
    low-T guard: the exponent (μ − E_D)/k_BT stays far below overflow across this
    bracket for T ≥ 30 K).

    Parameters
    ----------
    T : float
        Temperature in K.
    N_D : float
        Donor density, cm^-3.
    dE_D : float, optional
        Donor depth below E_c, eV (default 45 meV, P in Si).
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    tuple
        (mu, n, p): chemical potential (eV) and the two carrier densities (cm^-3).
    """
    kT = KB * T
    E_v = E_C - mat["Eg"]
    E_D = E_C - dE_D

    def imbalance(mu):
        ionized = N_D / (1.0 + G_D * np.exp((mu - E_D) / kT))
        return n_electrons(mu, T, mat) - p_holes(mu, T, mat) - ionized

    mu = brentq(imbalance, E_v + 1e-9, E_C + 0.3, xtol=1e-12)
    return mu, n_electrons(mu, T, mat), p_holes(mu, T, mat)


def takeover_temperature(N_D, mat=SI):
    """The intrinsic-takeover temperature: n_i(T) = N_D solved by scipy.optimize.brentq.

    Solved in the logarithm (ln n_i(T) − ln N_D) on the bracket [400, 1200] K —
    n_i is exponentially steep, and the log form keeps brentq's interpolation
    honest across the decades. This is Exercise 5's device-death solve.

    Parameters
    ----------
    N_D : float
        Donor density, cm^-3.
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Takeover temperature, K.
    """
    return brentq(
        lambda T: np.log(n_intrinsic(T, mat)) - np.log(N_D), 400.0, 1200.0, xtol=1e-6
    )


def builtin_potential(N_A, N_D, T=300.0, mat=SI):
    """The built-in potential qV_bi = k_BT ln(N_AN_D / n_i^2), in volts.

    eq-builtin-potential: one chemical potential must span a joined n-type and
    p-type region in equilibrium, so the bands bend by the difference of the bulk
    Boltzmann chemical potentials — one logarithm of mass action. Uses the exact
    n_intrinsic (brentq + Fermi–Dirac), not the closed form.

    Parameters
    ----------
    N_A : float
        Acceptor density on the p side, cm^-3.
    N_D : float
        Donor density on the n side, cm^-3.
    T : float, optional
        Temperature in K (default 300).
    mat : dict, optional
        Band-parameter set (default silicon).

    Returns
    -------
    float
        Built-in potential, V.
    """
    ni = n_intrinsic(T, mat)
    return KB * T * np.log(N_A * N_D / ni**2)


print(f"k_BT at 300 K = {KB * 300:.5f} eV (the 0.02585 of the conventions)")
print(
    f"N_c(Si, 300 K) = {N_eff(SI['mc'], 300):.3e} cm^-3,  N_v = {N_eff(SI['mv'], 300):.3e} cm^-3"
)

## Exercise 1 — Two bands, two species

The gapped DOS of [§7.12](bloch-theorem-band-structure.ipynb) meets the Fermi function of [§7.7](bose-einstein-fermi-dirac.ipynb) — electrons above, holes below, one
distribution. Cite {eq}`eq-two-bands`.

1. Write the carrier integrals $n = \int g_c\,n_F$ and $p = \int g_v(1 - n_F)$, with the hole
   bookkeeping derived from the filled-band-minus-occupations argument of [§7.12](bloch-theorem-band-structure.ipynb).
2. Implement `n_electrons` / `p_holes` via `scipy.integrate.quad` in the
   $x = (\varepsilon - E_c)/k_BT$ form (cutoff stated, insensitivity shown), and check the
   hole route against a brute-force integral in raw $\varepsilon$ with no substitution.
3. Plot the band diagram: $g(\varepsilon)$ with the gap, $\mu$, and the two shaded carrier
   populations at 300 K.
4. State the framing contract (prose, the course's voice): every tool in this notebook is
   second-hand (the function of [§7.7](bose-einstein-fermi-dirac.ipynb), the integrals of [§7.3](statistical-toolkit.ipynb), the license of [§7.8](classical-limit-thermal-wavelength.ipynb)), aimed at the object of [§7.12](bloch-theorem-band-structure.ipynb); the
   statistics resumes here and finishes the movement.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    cutoff_shift < 1e-10,
    "the F_1/2 cutoff is a non-issue: doubling x_cut moves n at the 1e-10 level or below",
    f"relative shift {cutoff_shift:.1e}",
)
validate.close(
    p_route2,
    p_route1,
    "the hole bookkeeping survives a second route: brute-force ε-integral vs the mirrored F_1/2",
    rtol=1e-8,
)
validate.check(
    n_test > 0 and p_route1 > 0 and np.isfinite(n_test) and np.isfinite(p_route1),
    "two carrier gases from one Fermi function, both evaluating stably at mid-gap μ",
    f"n = {n_test:.2e}, p = {p_route1:.2e} cm^-3",
)

## Exercise 2 — The engineer's constant is the thermal wavelength

The Boltzmann limit, its workhorse $N_c$, and the license quantified. Cite
{eq}`eq-boltzmann-carriers`.

1. Derive $n \to N_c e^{-(E_c-\mu)/k_BT}$ with $N_c = 2(m_ck_BT/2\pi\hbar^2)^{3/2}$, and prove
   the identity $N_c = 2/\lambda_T^3$ (the wavelength of [§7.8](classical-limit-thermal-wavelength.ipynb) for the carrier mass).
2. Verify the identity numerically to machine precision ($2.816\times10^{19}$ cm⁻³ by both
   routes, via `N_eff` and via `lambda_T`) and evaluate $N_v$.
3. Quantify the license against exact Fermi–Dirac: the ratio `n_boltzmann`/`n_electrons` at
   $E_c - \mu = 200, 50, -50$ meV — plot it against $(E_c - \mu)/k_BT$.
4. Read it (prose): nondegeneracy for carriers is verbatim $n\lambda^3 \ll 2$ — the movement's
   opening map ruling its closing notebook; the revocation at $n \sim N_c$ is Exercise 6's
   business.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    N_c_formula,
    N_c_wavelength,
    "N_c is 2/λ^3: the thermal wavelength in industrial dress, by two code routes",
    rtol=1e-10,
)
validate.close(
    ratios,
    [1.000, 1.050, 2.54],
    "the license quantified: exact in the tail, 5% at 2 k_BT, 2.5× wrong inside the band",
    rtol=2e-2,
)

## Exercise 3 — The law of mass action, and the silicon number

One line makes $np$ immune to doping; three constants make it $10^{10}$. Cite
{eq}`eq-mass-action`, {eq}`eq-intrinsic-density`.

1. Derive $np = N_cN_v e^{-E_g/k_BT} = n_i^2$ and state *why* $\mu$ cancels, hence doping-independence; note the equilibrium-constant structure (Saha named, one breath).
2. Compute $n_i(\text{Si}, 300\,\text{K})$ from the exact machinery (`n_intrinsic`) and
   compare with the accepted range ($9.65\times10^9$ modern to $1.45\times10^{10}$ in older
   texts, cited): within 10%, the residual honestly assigned to the effective-mass inputs,
   not tuned away.
3. Build the table: Ge and GaAs by the same three constants: seven decades from a
   factor-two gap spread; verify the sensitivity $d\ln n_i/dE_g = -1/2k_BT$ numerically from
   the exact solve at shifted gaps.
4. Weigh the scales (prose): one pair per $5.6\times10^{12}$ silicon atoms, a part-per-trillion impurity of pure thermodynamics, and the substrate of the information
   age; exponential gap-sensitivity stated as the field's defining fact.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    ni_Si,
    8.88e9,
    "the silicon number from ħ, one gap, two masses — cited against 9.65e9 accepted, not tuned",
    rtol=2e-2,
)
validate.close(
    [table["Ge"], table["GaAs"]],
    [1.5e13, 2.2e6],
    "the seven-decade table: Ge and GaAs from the same three constants each",
    rtol=5e-2,
)
validate.close(
    dlnni_dEg,
    -1.0 / (2.0 * kT0),
    "the exponential lever, measured from the exact solve: d ln n_i/dE_g = −1/2k_BT",
    rtol=1e-3,
)

## Exercise 4 — The chemical potential of an insulator

$\mu$ lives in the gap — where there are no states at all. Cite {eq}`eq-intrinsic-mu`.

1. Solve intrinsic neutrality $n = p$ exactly with `scipy.optimize.brentq` on the Fermi–Dirac
   integrals (bracket: the gap itself) at 300 and 500 K.
2. Derive the Boltzmann-level formula $\mu_i = (E_c + E_v)/2 + (3/4)k_BT\ln(m_v/m_c)$ and
   verify it matches the exact solve to all displayed digits.
3. Plot $\mu_i(T)$ and attribute the drift's sign (the heavier band wins states; $\mu$ leans
   away from it).
4. Address the scandal (prose): a chemical potential in a stateless region is routine for
   the formalism of [§7.7](bose-einstein-fermi-dirac.ipynb) and offensive to naive intuition; resolve it ($\mu$ is the bookkeeping
   price of a particle, not an address), and note the lineage of [§7.4](thermal-density-matrix.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [mu_exact_300, mu_exact_500],
    [mu_formula_300, mu_formula_500],
    "intrinsic μ: mid-gap plus the mass drift, matching the exact solve to all displayed digits",
    rtol=1e-4,
)

## Exercise 5 — Doping: the three regimes

Donor statistics, exact neutrality, and the temperature plateau on which every chip depends —
the movement's centerpiece. Cite {eq}`eq-doping-regimes`.

1. Implement `solve_neutrality(T, N_D, dE_D)`: $n = p + N_D^+$ with
   $N_D^+ = N_D/(1 + 2e^{(\mu - E_D)/k_BT})$ (the $g = 2$ stated; the bracket and its low-T
   guard stated), by `scipy.optimize.brentq`.
2. Sweep $T = 30$–800 K for $N_D = 10^{16}$ (P:Si, $\Delta E_D = 45$ meV) and reproduce the
   portrait: freeze-out, saturation across 100–450 K, intrinsic takeover; plot $\log n$ vs
   $1000/T$ with the regimes labelled and the freeze-out slope $\sim\Delta E_D/2$ exhibited
   (fit $\ln(n/T^{3/4})$ against $1/T$ over 30–45 K).
3. Solve the takeover: $n_i(T) = N_D$ by `brentq` in the logarithm on [400, 1200] K: device death, with the wide-gap moral (SiC/GaN, one breath outward); plot $\mu(T) - E_c$ sliding
   from the donor level toward mid-gap.
4. Work the seesaw (computation + prose): $p = n_i^2/n$ at 300 K, twelve decades of minority
   suppression from one doping decision; verify mass action at the doped $\mu$; read the
   plateau slowly.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [n_300, T_die],
    [9.96e15, 719.0],
    "the three regimes' landmarks: the plateau carrier density and device death",
    rtol=2e-2,
)
validate.close(
    [n_100 / N_D, n_450 / N_D],
    [0.68, 1.00],
    "the plateau bracketed: 68% ionized at 100 K, fully saturated at 450 K",
    rtol=5e-2,
)
validate.close(
    slope_fit,
    slope_theory,
    "freeze-out at the frozen slope ΔE_D/2 (prefactor-corrected fit over 30–45 K)",
    rtol=5e-2,
)
validate.close(
    p_300,
    7.9e3,
    "the seesaw: doping to 1e16 crushes minority holes twelve decades down",
    rtol=5e-2,
)
validate.close(
    mass_action_ratio,
    1.0,
    "and mass action holds at the doped μ: np = n_i^2, the potential genuinely cancelled",
    rtol=1e-3,
)

## Exercise 6 — The license revoked: the degenerate return

The number of [§7.8](classical-limit-thermal-wavelength.ipynb) climbs its ladder, Boltzmann fails on schedule, and the movement ends where it
began. Cite {eq}`eq-degenerate-return`.

1. Compute $n\lambda_T^3/2 = n/N_c$ across the doping ladder $10^{10} \to 10^{20}$ cm⁻³ (via
   `lambda_T`, so the identity does the work) and mark the crossover $n \sim N_c$.
2. Solve $\mu$ from exact neutrality at $N_D = 10^{20}$ by `scipy.optimize.brentq` on
   $n(\mu) = p(\mu) + N_D$ (donors taken fully ionized; the stated guard: at this density the
   donor levels merge into the band and the two-state statistics no longer applies) and confirm
   $\mu$ enters the conduction band.
3. Show Boltzmann's failure there quantitatively (Exercise 2's ratio at the degenerate $\mu$)
   and hand the regime to the formulas of [§7.9](fermi-gas-zero-temperature.ipynb): compute $\varepsilon_F - E_c$ for the carrier gas
   and its $T_F$.
4. Close the loop (prose): the heavily doped semiconductor is a metal by every measure the
   movement owns — degenerate emitters and ohmic contacts named as industry's daily use of
   exactly this limit; one dimensionless number opened Movement III and now closes it.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    ladder_vals,
    [3.6e-10, 3.6e-4, 0.99, 3.6],
    "the ladder verbatim: ten decades of classical headroom spent by one doping dial",
    rtol=5e-2,
)
validate.check(
    mu_heavy > E_C,
    "μ crosses into the conduction band at degenerate doping",
    f"μ − E_c = {mu_heavy - E_C:+.4f} eV",
)
validate.check(
    ratio_heavy > 2.0 and T_F > 300.0,
    "and Boltzmann fails there while T_F > T: the degenerate return to the metal of §7.9",
    f"Boltzmann/exact = {ratio_heavy:.2f}, T_F = {T_F:.0f} K",
)

## Exercise 7 — The 0.7 volts

One chemical potential, two doped regions, and the most famous number in hobby electronics.
Cite {eq}`eq-builtin-potential`.

1. State the equilibrium condition: joined materials share *one* $\mu$ (the [§7.4](thermal-density-matrix.ipynb)/[§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) lineage),
   so the bands must bend by the difference of the two bulk chemical potentials.
2. Derive $qV_{bi} = k_BT\ln(N_AN_D/n_i^2)$ from the Boltzmann carrier forms on each side.
3. Evaluate with `builtin_potential` for $N_A = N_D = 10^{16}$, the silicon diode drop;
   explore the (weak, logarithmic) doping dependence and the (strong) $n_i^2$ temperature
   dependence (why $V_{bi}$ sags with heat).
4. Defer outward (prose, per the framing constraint): depletion widths, rectification, and
   the transistor belong to device physics (Sze cited); this course computed the number and stops, because the statistics has done its job.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    V_bi,
    0.720,
    "the diode drop from one logarithm of mass action",
    rtol=1e-2,
)
validate.check(
    V_bi_T[400.0] < V_bi_T[300.0] < V_bi_T[250.0],
    "and it sags with heat, as every data sheet says: n_i^2 outruns the k_BT prefactor",
    f"{V_bi_T[250.0]:.3f} → {V_bi_T[400.0]:.3f} V across 250–400 K",
)

## Exercise 8 — Movement III, closed

The movement opened with a sea and closes with a gap, and the remarkable thing is how little
new physics the closing required. Five notebooks ago the free Fermi gas priced the heat
capacity and magnetism of real metals and weighed a dead star; then it confessed it could not
make an insulator; [§7.12](bloch-theorem-band-structure.ipynb) bought the one missing ingredient; and this notebook spent it, with tools the volume already owned. Fermi–Dirac in a gap gave a product law that doping cannot
touch, ten billion carriers per cubic centimetre from Planck's constant and one energy, a
three-hundred-kelvin plateau on which the information age operates, a seesaw that buries
minority carriers twelve decades deep, and — pushed hard enough — a return, on the
schedule of [§7.8](classical-limit-thermal-wavelength.ipynb), to the metal where it all began. The confession of [§7.12](bloch-theorem-band-structure.ipynb) is now fully answered:
copper and diamond and silicon differ not in their electrons but in their counting.

It is worth savoring that the most consequential materials in modern history are the
*mediocre* ones — neither the full bands of an insulator nor the half-filled abundance of a
metal, but a gap small enough to leak. Ten billion carriers per cubic centimetre is, by any
metallic standard, a rounding error. Civilization runs on the rounding error.

The movement's fermions now rest. Movement IV turns to the other statistics: light itself,
where the story the volume opened with (Planck, 1900) is finally told to its end ([§7.14](photon-gas-planck.ipynb)).

## Notebook summary

Movement III's finale, run entirely on second-hand tools, the notebook's stated point.

- **Two bands, two species** {eq}`eq-two-bands`: electrons by $n_F$ on the conduction √-edge,
  holes by $1 - n_F$ on the valence one (the filled-band bookkeeping of [§7.12](bloch-theorem-band-structure.ipynb) made statistical);
  exact evaluation by `quad` on the $F_{1/2}$ form, cutoff-insensitive at $10^{-10}$ and
  gated against a brute-force second route.
- **The engineer's constant unmasked** {eq}`eq-boltzmann-carriers`: $N_c \equiv 2/\lambda_T^3$
  at $10^{-10}$ by two code routes ($2.816\times10^{19}$ cm⁻³): the wavelength of [§7.8](classical-limit-thermal-wavelength.ipynb) in overalls;
  the license quantified (1.000 / 1.050 / 2.5 at 200 / 50 / −50 meV).
- **Mass action** {eq}`eq-mass-action`, {eq}`eq-intrinsic-density`: $\mu$ cancels, so doping
  cannot touch $np$; the silicon number $8.9\times10^9$ cm⁻³ within 10% of the cited accepted
  value; Ge/GaAs complete the seven-decade table; the lever measured at $-1/2k_BT$.
- **The stateless chemical potential** {eq}`eq-intrinsic-mu`: exact `brentq` neutrality vs
  mid-gap $+\,(3/4)k_BT\ln(m_v/m_c)$, matching to all displayed digits; the scandal resolved
  ($\mu$ is a price, not an address).
- **The three regimes** {eq}`eq-doping-regimes`: freeze-out at slope $\Delta E_D/2$
  (prefactor-corrected fit), the plateau ($n = N_D$ across 100–450 K: chemistry beating
  temperature, the premise of electronics), takeover at 719 K; the seesaw at
  $7.9\times10^3$ cm⁻³; mass action verified at the doped $\mu$.
- **The degenerate return** {eq}`eq-degenerate-return`: the $n\lambda^3/2$ ladder from
  $3.6\times10^{-10}$ to 3.6; $\mu$ crosses into the band; Boltzmann fails 3×; $T_F = 844$ K
  hands the gas to [§7.9](fermi-gas-zero-temperature.ipynb): the movement closed by the number that opened it.
- **The 0.7 volts** {eq}`eq-builtin-potential`: $V_{bi} = 0.720$ V from one logarithm, sagging
  with heat as every data sheet says; the junction's mechanics deferred outward.

Movement IV opens next door with the photon gas.

## Outlook

- **Movement IV: the other statistics.** The photon gas and Planck's law: the crisis that opened the volume's story (1900), resolved with the volume's own tools ([§7.14](photon-gas-planck.ipynb)); phonons on the Bloch counting of [§7.12](bloch-theorem-band-structure.ipynb) ([§7.16](phonons-debye.ipynb)); condensation
  when the boson ceiling saturates ([§7.17](bose-einstein-condensation.ipynb)).
- **Devices, outward.** The p–n junction's mechanics, the transistor, LEDs as mass action run
  backwards (Sze & Ng); SiC and GaN on the wide-gap frontier.
- **Real gaps and masses.** From electronic structure: DFT and the MMM course, the standing bridge for everything this notebook took as stated inputs.
- **The Saha equation.** This notebook's mathematics in stellar atmospheres: ionization equilibrium as mass action with the vacuum as a band.
- **Cross-reference** [§7.7](bose-einstein-fermi-dirac.ipynb) (the function, unchanged), [§7.3](statistical-toolkit.ipynb) (the integrals), [§7.8](classical-limit-thermal-wavelength.ipynb) (the license,
  twice decisive), [§7.9](fermi-gas-zero-temperature.ipynb) (the degenerate return), [§7.12](bloch-theorem-band-structure.ipynb) (the object consumed), [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (Pauli
  underneath it all).

In [ ]:
from ecp.style import footer

footer()